# Test minimization

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "NfaToDfa.fsx"
#load "Minimize.fsx"
#load "FsmToDot.fsx"

In [ ]:
open AutomataTypes
open NfaToDfa
open Minimize
open FsmToDot

The nine-state example is the same as in the original notebook. States 4, 5 and 8 are unreachable from the start state and disappear before equivalent states are grouped.

In [ ]:
let delta =
    [ (0, 'a'), 1; (0, 'b'), 6
      (1, 'a'), 2; (1, 'b'), 7
      (2, 'a'), 3; (2, 'b'), 6
      (3, 'a'), 0; (3, 'b'), 7
      (4, 'a'), 5; (4, 'b'), 2
      (5, 'a'), 6; (5, 'b'), 8
      (6, 'a'), 7; (6, 'b'), 2
      (7, 'a'), 0; (7, 'b'), 3
      (8, 'a'), 4; (8, 'b'), 8 ] |> Map.ofList
let automaton : Dfa<int> =
    { States = set [0..8]; Alphabet = set ['a';'b']; Transitions = delta
      Start = 0; Accepting = set [2;3;6;7] }
let originalDot, _ = dfa2dot automaton
originalDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg originalDot), "text/html") |> ignore

In [ ]:
let minimized = minimize automaton
let minimizedDot, stateNames = dfa2dot minimized
minimizedDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg minimizedDot), "text/html") |> ignore

In [ ]:
stateNames

The four classes are `{0}`, `{1}`, `{2,6}` and `{3,7}`. Check the partition and compare acceptance on all words of length at most eight.

In [ ]:
checkNotebook (minimized.States = set [set [0]; set [1]; set [2;6]; set [3;7]])
let rec words length =
    if length = 0 then [""]
    else [ for prefix in words (length-1) do for c in automaton.Alphabet do yield prefix + string c ]
for length in 0 .. 8 do
    for word in words length do
        checkNotebook (acceptsDfa automaton word = acceptsDfa minimized word)
let partial = { automaton with Transitions = Map.remove (0, 'a') automaton.Transitions }
let rejected = try minimize partial |> ignore; false with :? System.ArgumentException -> true
checkNotebook rejected